# KRK Fault SP vs DP

## Run C++ examples

In [1]:
%%bash
TOP=${TOP:-$(git rev-parse --show-toplevel)}
PATH=${TOP}/build/dpsim/examples/cxx

#1ms simulation
DURATION=12
STARTTIMEFAULT=10
ENDTIMEFAULT=10.2
TIMESTEP=50e-6
TIMESTEPSTR=$(printf "%1.6f\n" ${TIMESTEP})

delays_zoh=(100e-6, 200e-6, 0.5e-3, 1e-3)

prefixes_zoh=(zoh_100e-6 zoh_200e-6 zoh_0.5e-3 zoh_1e-3)

SP_SynGenTrStab_KRK_TwoArea_SteadyState -d ${DURATION}

for i in "${!delays_zoh[@]}"; do
    SP_SynGenTrStab_KRK_TwoArea_SteadyState_split_ITM -t ${TIMESTEP} -d ${DURATION} -o delay="${delays_zoh[$i]}" -o method=extrapolation-zoh -o prefix="${prefixes_zoh[$i]}"
done

In [2]:
# Read results
import villas.dataprocessing.readtools as rt
from villas.dataprocessing.timeseries import TimeSeries as ts
import matplotlib.pyplot as plt
import re
import numpy as np
import math
import os
from datetime import date
import pandas as pd
from collections import defaultdict


%matplotlib widget

In [ ]:
V_nom = 230e3

work_dir = 'logs/SP_SynGenTrStab_KRK_TwoArea_SteadyState_PF/'
log_name = 'SP_SynGenTrStab_KRK_TwoArea_SteadyState_PF'

print(work_dir + log_name + '.csv')

ts_pfsimpy = rt.read_timeseries_dpsim(work_dir + log_name + '.csv')
results = pd.DataFrame(columns=['Bus', 'V Mag. [V]', 'V rel Ang. [deg]', 'V Ang. [deg]', 'P', 'Q'])

s_prefix = 's_'
v_prefix = 'v_'

vble_result_columns_abs = defaultdict(lambda: [0, 0, 0, 0, 0])
for column in ts_pfsimpy.keys():
    if column.startswith(v_prefix):
        column_base = column.replace(v_prefix, '')
        vble_result_columns_abs[column_base][0] = "{0:.2f}".format(np.absolute(ts_pfsimpy[column].values[-1])/V_nom)
        vble_result_columns_abs[column_base][1] = "{0:.2f}".format(np.degrees(np.angle(ts_pfsimpy[column].values[-1])))
        vble_result_columns_abs[column_base][2] = "{0:.2f}".format(np.degrees(np.angle(ts_pfsimpy[column].values[-1])) - 6.8)
    else:
        column_base = column.replace(s_prefix, '')
        vble_result_columns_abs[column_base][3] = "{0:.2e}".format(np.real(ts_pfsimpy[column].values[-1]))
        vble_result_columns_abs[column_base][4] = "{0:.2e}".format(np.imag(ts_pfsimpy[column].values[-1]))

i = 0
for node,node_data in vble_result_columns_abs.items():
    results.loc[i] = [node] + node_data
    i += 1

print(results)

## Results 1ph SP

In [ ]:
work_dir = 'logs/SP_SynGenTrStab_KRK_TwoArea_SteadyState_SP/'
log_name = 'SP_SynGenTrStab_KRK_TwoArea_SteadyState_SP'
print(work_dir + log_name + '.csv')
ts_sp1ph_TrStab= rt.read_timeseries_dpsim(work_dir + log_name + '.csv')

In [ ]:
H_v = np.array([100e-6, 200e-6, 0.5e-3, 1e-3])
H_v_legends = ["100e-6", "200e-6", "0.5e-3", "1e-3"]

ts_sp1ph_TrStab_itm_delay = {}
ts_sp1ph_TrStab_itm_zoh = {}

for H_v_leg in H_v_legends:
    ts_sp1ph_TrStab_itm_delay[H_v_leg] = rt.read_timeseries_dpsim(
        "logs/SP_SynGenTrStab_KRK_TwoArea_SteadyState_split_ITM_delay_"
        + H_v_leg
        + "_SP/SP_SynGenTrStab_KRK_TwoArea_SteadyState_split_ITM_delay_"
        + H_v_leg
        + "_SP.csv"
    )

    ts_sp1ph_TrStab_itm_zoh[H_v_leg] = rt.read_timeseries_dpsim(
        "logs/SP_SynGenTrStab_KRK_TwoArea_SteadyState_split_ITM_zoh_"
        + H_v_leg
        + "_SP/SP_SynGenTrStab_KRK_TwoArea_SteadyState_split_ITM_zoh_"
        + H_v_leg
        + "_SP.csv"
    )

## Parameters

In [6]:
timestep=50e-6
t_begin=0
t_end=12

begin_idx = int(t_begin/timestep)
end_idx= int(t_end/timestep)

## Interface node - N8

In [ ]:
name_monolithic = 'v8'
name_cosim = 'v8_1'
plt.plot(
    ts_sp1ph_TrStab[name_monolithic].interpolate(timestep).time[begin_idx:end_idx],
    1e-3
    * ts_sp1ph_TrStab[name_monolithic]
    .interpolate(timestep)
    .frequency_shift(60)
    .values[begin_idx:end_idx],
    label="Monolithic",
)

line_style = ['-', '--', '-.', ':']

for plot_series in H_v_legends:
    plt.plot(
        ts_sp1ph_TrStab_itm_delay[plot_series][name_cosim]
        .interpolate(timestep)
        .time[begin_idx:end_idx],
        1e-3
        * ts_sp1ph_TrStab_itm_delay[plot_series][name_cosim]
        .interpolate(timestep)
        .frequency_shift(60)
        .values[begin_idx:end_idx],
        label="Delay",
    )
    plt.plot(
        ts_sp1ph_TrStab_itm_zoh[plot_series][name_cosim]
        .interpolate(timestep)
        .time[begin_idx:end_idx],
        1e-3
        * ts_sp1ph_TrStab_itm_zoh[plot_series][name_cosim]
        .interpolate(timestep)
        .frequency_shift(60)
        .values[begin_idx:end_idx],
        line_style[H_v_legends.index(plot_series)],
        label="Cosim." + ", H=" + plot_series
    )
plt.legend(loc="lower right")
plt.grid()
plt.tight_layout()
plt.xlabel("time (s)")
plt.ylabel("voltage (kV)")
plt.title("N8")

## Generator terminal voltage

In [8]:
# plt.subplot(2, 2, 1)
# for name in ['v_gen1']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], label='N1')
#     plt.plot(ts_sp1ph_TrStab_dl_itm[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl_itm[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], '--', label='N1 cosim.')
# plt.legend(loc='lower right')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('voltage (kV)')
# plt.title('Gen. 1')

# plt.subplot(2, 2, 2)
# for name in ['v_gen2']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], label='N2')
#     plt.plot(ts_sp1ph_TrStab_dl_itm[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl_itm[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], '--', label='N2 cosim.')
# plt.legend(loc='lower right')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('voltage (kV)')
# plt.title('Gen. 2')

# plt.subplot(2, 2, 3)
# for name in ['v_gen3']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], label='N3')
#     plt.plot(ts_sp1ph_TrStab_dl_itm[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl_itm[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], '--', label='N3 cosim.')
# plt.legend(loc='lower right')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('voltage (kV)')
# plt.title('Gen. 3')

# plt.subplot(2, 2, 4)
# for name in ['v_gen4']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], label='N4')
#     plt.plot(ts_sp1ph_TrStab_dl_itm[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl_itm[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], '--', label='N4 cosim.')
# plt.legend(loc='lower right')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('voltage (kV)')
# plt.title('Gen. 4')


## Generator terminal Current

In [9]:
# plt.figure()

# plt.subplot(2,2,1)
# for name in ['i_gen1']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], label='DP', linestyle='--')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('current (kA)')
# plt.title('Gen. 1')

# plt.subplot(2,2,2)
# for name in ['i_gen2']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], label='DP', linestyle='--')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('current (kA)')
# plt.title('Gen. 2')

# plt.subplot(2,2,3)
# for name in ['i_gen3']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], label='DP', linestyle='--')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('current (kA)')
# plt.title('Gen. 3')

# plt.subplot(2,2,4)
# for name in ['i_gen4']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], label='DP', linestyle='--')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('current (kA)')
# plt.title('Gen. 4')


## Line currents

In [10]:
# plt.figure()

# plt.subplot(2,2,1)
# for name in ['i_line56']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], label='i line 56')
    
# # plt.legend(loc='lower right')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('current (kA)')
# plt.title('i line 56')

# plt.subplot(2,2,2)
# for name in ['i_line67']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], label='i line 67')
    
# # plt.legend(loc='lower right')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('current (kA)')
# plt.title('i line 67')

# plt.subplot(2,2,3)
# for name in ['i_line1011']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], label='i line 1011')
    
# # plt.legend(loc='lower right')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('current (kA)')
# plt.title('i line 1011')

# plt.subplot(2,2,4)
# for name in ['i_line910']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-3*ts_sp1ph_TrStab_dl[name].interpolate(timestep).frequency_shift(60).values[begin_idx:end_idx], label='i line 910')
    
# # plt.legend(loc='lower right')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('current (kA)')
# plt.title('i line 910')

## Interface line currents

In [ ]:
plt.figure()

# plt.subplot(1, 2, 1)
plt.plot(
    ts_sp1ph_TrStab["i_line78_1"].interpolate(timestep).time[begin_idx:end_idx],
    1e-3
    * ts_sp1ph_TrStab["i_line78_1"]
    .interpolate(timestep)
    .frequency_shift(60)
    .values[begin_idx:end_idx],
    label="Monolithic",
)

# plt.subplot(1, 2, 2)
# plt.plot(
#     ts_sp1ph_TrStab["i_line89_1"].interpolate(timestep).time[begin_idx:end_idx],
#     1e-3
#     * ts_sp1ph_TrStab["i_line89_1"]
#     .interpolate(timestep)
#     .frequency_shift(60)
#     .values[begin_idx:end_idx],
#     label="Monolithic",
# )

for plot_series in H_v_legends:
    # plt.plot(
    #     ts_sp1ph_TrStab_itm_delay[H_v_legends[0]][name]
    #     .interpolate(timestep)
    #     .time[begin_idx:end_idx],
    #     1e-3
    #     * ts_sp1ph_TrStab_itm_delay[H_v_legends[0]][name]
    #     .interpolate(timestep)
    #     .frequency_shift(60)
    #     .values[begin_idx:end_idx],
    #     label="Delay",
    # )
    # plt.subplot(1, 2, 1)
    plt.plot(
        ts_sp1ph_TrStab_itm_zoh[plot_series]["i_line78_1"]
        .interpolate(timestep)
        .time[begin_idx:end_idx],
        1e-3
        * ts_sp1ph_TrStab_itm_zoh[plot_series]["i_line78_1"]
        .interpolate(timestep)
        .frequency_shift(60)
        .values[begin_idx:end_idx],
        line_style[H_v_legends.index(plot_series)],
        label="Cosim." + ", H=" + plot_series
    )
    # plt.legend(loc='lower right')
    plt.grid()
    plt.tight_layout()
    plt.xlabel("time (s)")
    plt.ylabel("current (kA)")
    plt.title("i line 78_1")
    plt.legend(loc="lower right")

    # plt.subplot(1, 2, 2)
    # plt.plot(
    #     ts_sp1ph_TrStab_itm_delay[H_v_legends[0]]["i_line89_1"]
    #     .interpolate(timestep)
    #     .time[begin_idx:end_idx],
    #     1e-3
    #     * ts_sp1ph_TrStab_itm_delay[H_v_legends[0]]["i_line89_1"]
    #     .interpolate(timestep)
    #     .frequency_shift(60)
    #     .values[begin_idx:end_idx],
    #     label="Delay",
    # )
    # plt.plot(
    #     ts_sp1ph_TrStab_itm_zoh[plot_series]["i_line89_1"]
    #     .interpolate(timestep)
    #     .time[begin_idx:end_idx],
    #     1e-3
    #     * ts_sp1ph_TrStab_itm_zoh[plot_series]["i_line89_1"]
    #     .interpolate(timestep)
    #     .frequency_shift(60)
    #     .values[begin_idx:end_idx],
    #     line_style[H_v_legends.index(plot_series)],
    #     label="Cosim." + ", H=" + plot_series
    # )
    # # plt.legend(loc='lower right')
    # plt.grid()
    # plt.tight_layout()
    # plt.xlabel("time (s)")
    # plt.ylabel("current (kA)")
    # plt.title("i line 89_1")
    # plt.legend(loc="lower right")

## Generator electrical & mechanical energy

In [12]:
# plt.figure()
    
# plt.subplot(2,2,1)
# for name in ['P_elec1']:    
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-6*ts_sp1ph_TrStab_dl[name].interpolate(timestep).values[begin_idx:end_idx], label='P_elec G1')
# # plt.legend(loc='lower right')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('power (MW)')
# plt.title('P_elec G1')

# plt.subplot(2,2,2)
# for name in ['P_elec2']:    
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-6*ts_sp1ph_TrStab_dl[name].interpolate(timestep).values[begin_idx:end_idx], label='P_elec G2')

# # plt.legend(loc='lower right')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('power (MW)')
# plt.title('P_elec G2')

# plt.subplot(2,2,3)
# for name in ['P_elec3']:    
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-6*ts_sp1ph_TrStab_dl[name].interpolate(timestep).values[begin_idx:end_idx], label='P_elec G3')
# # plt.legend(loc='lower right')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('power (MW)')
# plt.title('P_elec G3')

# plt.subplot(2,2,4)
# for name in ['P_elec4']:    
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx], 1e-6*ts_sp1ph_TrStab_dl[name].interpolate(timestep).values[begin_idx:end_idx], label='P_elec G4')
# # plt.legend(loc='lower right')
# plt.grid()
# plt.tight_layout()
# plt.xlabel('time (s)')
# plt.ylabel('power (MW)')
# plt.title('P_elec G4')

## Rotor angular speed $\omega_r$

In [ ]:
plt.figure()
plt.plot(
    ts_sp1ph_TrStab["wr_gen1"].interpolate(timestep).time[begin_idx:end_idx],
    ts_sp1ph_TrStab["wr_gen1"].interpolate(timestep).values[begin_idx:end_idx]
    * 60
    / 377,
    label="Monolithic",
)
# plt.legend(loc='lower right')
plt.xlabel("time (s)")
plt.ylabel("Frequency (Hz)")
plt.grid()
plt.tight_layout()
plt.title("Generator G1, $\omega_{r,1}$")

for plot_series in H_v_legends:
    plt.plot(
        ts_sp1ph_TrStab_itm_zoh[plot_series]["wr_gen1"].interpolate(timestep).time[begin_idx:end_idx],
        ts_sp1ph_TrStab_itm_zoh[plot_series]["wr_gen1"].interpolate(timestep).values[begin_idx:end_idx]
        * 60
        / 377,
        line_style[H_v_legends.index(plot_series)],
        label="Cosim." + ", H=" + plot_series
    )
    plt.legend(loc="lower right")

In [ ]:
plt.figure()
plt.plot(
    ts_sp1ph_TrStab["wr_gen2"].interpolate(timestep).time[begin_idx:end_idx],
    ts_sp1ph_TrStab["wr_gen2"].interpolate(timestep).values[begin_idx:end_idx]
    * 60
    / 377,
    label="Monolithic",
)
# plt.legend(loc='lower right')
plt.xlabel("time (s)")
plt.ylabel("Frequency (Hz)")
plt.grid()
plt.tight_layout()
plt.title("Generator G2, $\omega_{r,2}$")

for plot_series in H_v_legends:
    plt.plot(
        ts_sp1ph_TrStab_itm_zoh[plot_series]["wr_gen2"].interpolate(timestep).time[begin_idx:end_idx],
        ts_sp1ph_TrStab_itm_zoh[plot_series]["wr_gen2"].interpolate(timestep).values[begin_idx:end_idx]
        * 60
        / 377,
        line_style[H_v_legends.index(plot_series)],
        label="Cosim." + ", H=" + plot_series
    )
    plt.legend(loc="lower right")

In [ ]:
plt.figure()
plt.plot(
    ts_sp1ph_TrStab["wr_gen3"].interpolate(timestep).time[begin_idx:end_idx],
    ts_sp1ph_TrStab["wr_gen3"].interpolate(timestep).values[begin_idx:end_idx]
    * 60
    / 377,
    label="Monolithic",
)
# plt.legend(loc='lower right')
plt.xlabel("time (s)")
plt.ylabel("Frequency (Hz)")
plt.grid()
plt.tight_layout()
plt.title("Generator G3, $\omega_{r,2}$")

for plot_series in H_v_legends:
    plt.plot(
        ts_sp1ph_TrStab_itm_zoh[plot_series]["wr_gen3"].interpolate(timestep).time[begin_idx:end_idx],
        ts_sp1ph_TrStab_itm_zoh[plot_series]["wr_gen3"].interpolate(timestep).values[begin_idx:end_idx]
        * 60
        / 377,
        line_style[H_v_legends.index(plot_series)],
        label="Cosim." + ", H=" + plot_series
    )
    plt.legend(loc="lower right")

In [ ]:
plt.figure()
plt.plot(
    ts_sp1ph_TrStab["wr_gen4"].interpolate(timestep).time[begin_idx:end_idx],
    ts_sp1ph_TrStab["wr_gen4"].interpolate(timestep).values[begin_idx:end_idx]
    * 60
    / 377,
    label="Monolithic",
)
# plt.legend(loc='lower right')
plt.xlabel("time (s)")
plt.ylabel("Frequency (Hz)")
plt.grid()
plt.tight_layout()
plt.title("Generator G4, $\omega_{r,2}$")

for plot_series in H_v_legends:
    plt.plot(
        ts_sp1ph_TrStab_itm_zoh[plot_series]["wr_gen4"].interpolate(timestep).time[begin_idx:end_idx],
        ts_sp1ph_TrStab_itm_zoh[plot_series]["wr_gen4"].interpolate(timestep).values[begin_idx:end_idx]
        * 60
        / 377,
        line_style[H_v_legends.index(plot_series)],
        label="Cosim." + ", H=" + plot_series
    )
    plt.legend(loc="lower right")

## Rotor angle $\delta _r$

In [17]:
# plt.figure()

# plt.subplot(2,2,1)
# for name in ['delta_gen1']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx],
#         ts_sp1ph_TrStab_dl[name].interpolate(timestep).values[begin_idx:end_idx]*180/3.14 - ts_sp1ph_TrStab_dl['deltaref_gen1'].interpolate(timestep).values[begin_idx:end_idx]*180/3.14, label='SP')
# # plt.legend(loc='lower right')
# plt.xlabel('time (s)')
# plt.ylabel('angle (°)')
# plt.grid()
# plt.tight_layout()
# plt.title('$\delta_1$')

# plt.subplot(2,2,2)
# for name in ['delta_gen2']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx],
#         ts_sp1ph_TrStab_dl[name].interpolate(timestep).values[begin_idx:end_idx]*180/3.14 - ts_sp1ph_TrStab_dl['deltaref_gen2'].interpolate(timestep).values[begin_idx:end_idx]*180/3.14, label='SP')
# # plt.legend(loc='lower right')
# plt.xlabel('time (s)')
# plt.ylabel('angle (°)')
# plt.grid()
# plt.tight_layout()
# plt.title('$\delta_2$')

# plt.subplot(2,2,3)
# for name in ['delta_gen3']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx],
#         ts_sp1ph_TrStab_dl[name].interpolate(timestep).values[begin_idx:end_idx]*180/3.14 - ts_sp1ph_TrStab_dl['delta_gen3'].interpolate(timestep).values[begin_idx:end_idx]*180/3.14, label='SP')
# # plt.legend(loc='lower right')
# plt.xlabel('time (s)')
# plt.ylabel('angle (°)')
# plt.grid()
# plt.tight_layout()
# plt.title('$\delta_3$')

# plt.subplot(2,2,4)
# for name in ['delta_gen4']:
#     plt.plot(ts_sp1ph_TrStab_dl[name].interpolate(timestep).time[begin_idx:end_idx],
#         ts_sp1ph_TrStab_dl[name].interpolate(timestep).values[begin_idx:end_idx]*180/3.14 - ts_sp1ph_TrStab_dl['deltaref_gen4'].interpolate(timestep).values[begin_idx:end_idx]*180/3.14, label='SP')
# # plt.legend(loc='lower right')
# plt.xlabel('time (s)')
# plt.ylabel('angle (°)')
# plt.grid()
# plt.tight_layout()
# plt.title('$\delta_4$')


## Exchanged power

In [ ]:
power_line3 = (
    np.conjugate(ts_sp1ph_TrStab["i_line78_1"].interpolate(timestep).values)
    * ts_sp1ph_TrStab["v7"].interpolate(timestep).values
)
power_line4 = (
    np.conjugate(ts_sp1ph_TrStab["i_line78_2"].interpolate(timestep).values)
    * ts_sp1ph_TrStab["v7"].interpolate(timestep).values
)
power_line5 = (
    np.conjugate(ts_sp1ph_TrStab["i_line89_1"].interpolate(timestep).values)
    * ts_sp1ph_TrStab["v8"].interpolate(timestep).values
)
power_line6 = (
    np.conjugate(ts_sp1ph_TrStab["i_line89_2"].interpolate(timestep).values)
    * ts_sp1ph_TrStab["v8"].interpolate(timestep).values
)

plt.figure()
plt.plot(
    ts_sp1ph_TrStab["i_line78_1"].interpolate(timestep).time,
    (power_line3 + power_line4) * 1e-6,
)

# power_line3_itm_delay = (
#     np.conjugate(
#         ts_sp1ph_TrStab_itm_delay[H_v_legends[0]]["i_line78_1"]
#         .interpolate(timestep)
#         .values
#     )
#     * ts_sp1ph_TrStab_itm_delay[H_v_legends[0]]["v7"].interpolate(timestep).values
# )
# power_line4_itm_delay = (
#     np.conjugate(
#         ts_sp1ph_TrStab_itm_delay[H_v_legends[0]]["i_line78_2"]
#         .interpolate(timestep)
#         .values
#     )
#     * ts_sp1ph_TrStab_itm_delay[H_v_legends[0]]["v7"].interpolate(timestep).values
# )
# power_line5_itm_delay = (
#     np.conjugate(
#         ts_sp1ph_TrStab_itm_delay[H_v_legends[0]]["i_line89_1"]
#         .interpolate(timestep)
#         .values
#     )
#     * ts_sp1ph_TrStab_itm_delay[H_v_legends[0]]["v8_1"].interpolate(timestep).values
# )
# power_line6_itm_delay = (
#     np.conjugate(
#         ts_sp1ph_TrStab_itm_delay[H_v_legends[0]]["i_line89_2"]
#         .interpolate(timestep)
#         .values
#     )
#     * ts_sp1ph_TrStab_itm_delay[H_v_legends[0]]["v8_1"].interpolate(timestep).values
# )

for plot_series in H_v_legends:
    power_line3_itm_zoh = (
        np.conjugate(
            ts_sp1ph_TrStab_itm_zoh[plot_series]["i_line78_1"]
            .interpolate(timestep)
            .values
        )
        * ts_sp1ph_TrStab_itm_zoh[plot_series]["v7"].interpolate(timestep).values
    )
    power_line4_itm_zoh = (
        np.conjugate(
            ts_sp1ph_TrStab_itm_zoh[plot_series]["i_line78_2"]
            .interpolate(timestep)
            .values
        )
        * ts_sp1ph_TrStab_itm_zoh[plot_series]["v7"].interpolate(timestep).values
    )
    power_line5_itm_zoh = (
        np.conjugate(
            ts_sp1ph_TrStab_itm_zoh[plot_series]["i_line89_1"]
            .interpolate(timestep)
            .values
        )
        * ts_sp1ph_TrStab_itm_zoh[plot_series]["v8_1"].interpolate(timestep).values
    )
    power_line6_itm_zoh = (
        np.conjugate(
            ts_sp1ph_TrStab_itm_zoh[plot_series]["i_line89_2"]
            .interpolate(timestep)
            .values
        )
        * ts_sp1ph_TrStab_itm_zoh[plot_series]["v8_1"].interpolate(timestep).values
    )
    # plt.plot(
    #     ts_sp1ph_TrStab_itm_delay[H_v_legends[0]]["i_line78_1"].interpolate(timestep).time,
    #     (power_line3_itm_delay + power_line4_itm_delay) * 1e-6,
    # )
    plt.plot(
        ts_sp1ph_TrStab_itm_zoh[H_v_legends[0]]["i_line78_1"].interpolate(timestep).time,
        (power_line3_itm_zoh + power_line4_itm_zoh) * 1e-6,
        line_style[H_v_legends.index(plot_series)],
        label="Cosim." + ", H=" + plot_series
    )
    plt.legend(loc="lower right")
    # plt.plot(
    #     ts_sp1ph_TrStab["i_line89_1"].interpolate(timestep).time,
    #     (power_line5 + power_line6) * 1e-6,
    # )
    # plt.plot(
    #     ts_sp1ph_TrStab_itm_delay[H_v_legends[0]]["i_line89_1"].interpolate(timestep).time,
    #     (power_line5_itm_delay + power_line6_itm_delay) * 1e-6,
    #     "--",
    # )
    # plt.plot(
    #     ts_sp1ph_TrStab_itm_zoh[H_v_legends[0]]["i_line89_1"].interpolate(timestep).time,
    #     (power_line5_itm_zoh + power_line6_itm_zoh) * 1e-6,
    #     "-.",
    # )
plt.xlabel("time [s]")
plt.ylabel("Power [MW]")
plt.grid()